# LAB DAY 19: Xây dựng hệ thống GraphRAG với Tech Company Corpus

**Sinh viên:** Nguyen Hoang Minh - 2A202600963

## Mục tiêu
1. Trích xuất Entity & Relation từ văn bản thô
2. Xây dựng đồ thị tri thức (NetworkX / Neo4j / NodeRAG)
3. Truy vấn đa bước (Multi-hop BFS)
4. So sánh **Flat RAG (ChromaDB)** vs **GraphRAG (NetworkX + BFS)** trên 20 câu hỏi benchmark

## Kết quả chính (Full LLM)
- Flat RAG: **100%** | GraphRAG: **90%** | Multi-hop Graph: **93.3%**
- Demo UI: `streamlit run streamlit_app.py`

## PHẦN 1: Nghiên cứu

### Entity Extraction: Phân biệt Node và Thuộc tính?
- **Node**: thực thể có thể kết nối nhiều quan hệ (OpenAI, Sam Altman)
- **Thuộc tính**: mô tả gắn với 1 node (không tạo node riêng nếu không cần multi-hop)
- LLM dùng schema quan hệ (FOUNDED_BY, CEO_OF) để chuẩn hóa

### Deduplication quan trọng vì?
- Tránh node trùng ("Google" vs "google")
- Giữ đồ thị gọn, BFS chính xác hơn

### BFS vs Vector Search?
- **Vector**: similarity ngữ nghĩa, tốt cho retrieval đơn đoạn
- **BFS**: duyệt theo cạnh, tốt cho multi-hop (DeepMind → Google → CEO)

In [ ]:
# Setup
import os
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

PROJECT_ROOT = Path(".").resolve()
os.chdir(PROJECT_ROOT)

# Set API key: os.environ["OPENAI_API_KEY"] = "sk-..."
from dotenv import load_dotenv
load_dotenv()

print("API key set:", bool(os.getenv("OPENAI_API_KEY")))

## Bước 1: Entity & Relation Extraction (Indexing)

In [ ]:
from src.config import CORPUS_PATH, TRIPLES_PATH
from src.entity_extraction import extract_triples_from_corpus, save_triples

# Đọc corpus
corpus = CORPUS_PATH.read_text(encoding="utf-8")
print(corpus[:500], "...")

# Trích xuất triples (demo=True nếu chưa có API key)
demo = not os.getenv("OPENAI_API_KEY")
result = extract_triples_from_corpus(CORPUS_PATH, demo=demo)
save_triples(result.triples, TRIPLES_PATH)

print(f"\nExtracted {len(result.triples)} triples ({result.source})")
print("Sample triples:")
for t in result.triples[:5]:
    print(f"  {t}")

## Bước 2: Graph Construction (NetworkX)

In [ ]:
from src.graph_construction import build_networkx_graph, graph_stats, get_neighbors_bfs
from src.visualize import visualize_graph
from src.config import GRAPH_IMAGE_PATH

graph = build_networkx_graph(result.triples)
stats = graph_stats(graph)
print("Graph stats:", stats)

visualize_graph(graph, GRAPH_IMAGE_PATH)
display(Image(filename=str(GRAPH_IMAGE_PATH)))

## Bước 3: Querying (GraphRAG - BFS 2-hop)

In [ ]:
from src.querying import answer_with_graph
from src.flat_rag import FlatRAG

question = "Ai là CEO của công ty đã mua DeepMind?"
graph_ans = answer_with_graph(question, graph)
flat_rag = FlatRAG(CORPUS_PATH)
flat_rag.index()
flat_ans = flat_rag.answer(question)

print("Question:", question)
print("\n--- GraphRAG ---")
print(graph_ans.answer)
print("\n--- Flat RAG ---")
print(flat_ans.answer)

## Bước 4: Evaluation — 20 câu hỏi benchmark

### Kết quả (Full LLM — chạy qua Streamlit)

| Metric | Flat RAG (ChromaDB) | GraphRAG (NetworkX + BFS) |
|--------|---------------------|---------------------------|
| Overall accuracy | **100.0%** | **90.0%** |
| Multi-hop accuracy | **100.0%** | **93.3%** |
| Graph wins (Flat wrong) | — | **0** |
| Avg latency | 2.07s | 2.00s |
| Eval tokens | 5,931 | 9,650 |

**GraphRAG sai 2 câu:** Q5 (thiếu số tiền đầu tư trong đồ thị), Q17 (thiếu giá mua Twitter).

**Không có trường hợp** GraphRAG sửa lỗi ảo giác của Flat RAG trong lần chạy này.

In [ ]:
from src.evaluation import run_evaluation, print_summary
from src.config import EVAL_RESULTS_PATH, COST_REPORT_PATH

df = run_evaluation(graph, flat_rag)
print_summary(df)
df

## Phân tích chi phí (Token & Time)

In [ ]:
from src.config import COST_REPORT_PATH, EVAL_RESULTS_PATH
from src.evaluation import compute_summary

# Đọc kết quả đã lưu (từ Streamlit hoặc python main.py)
if EVAL_RESULTS_PATH.exists():
    df = pd.read_csv(EVAL_RESULTS_PATH)
    summary = compute_summary(df)
    print("=== BENCHMARK SUMMARY ===")
    for k, v in summary.items():
        print(f"  {k}: {v}")

if COST_REPORT_PATH.exists():
    print("\n=== COST ANALYSIS ===")
    print(json.dumps(json.loads(COST_REPORT_PATH.read_text()), indent=2, ensure_ascii=False))

# GraphRAG đúng khi Flat RAG sai
graph_fixes = df[(df["flat_correct"] == "Sai") & (df["graph_correct"] == "Đúng")]
print(f"\nGraphRAG fixes Flat RAG errors: {len(graph_fixes)} cases")
if len(graph_fixes):
    display(graph_fixes[["id", "question", "flat_rag_answer", "graph_rag_answer"]])

# GraphRAG sai khi Flat RAG đúng
flat_wins = df[(df["flat_correct"] == "Đúng") & (df["graph_correct"] == "Sai")]
print(f"\nFlat RAG wins over GraphRAG: {len(flat_wins)} cases")
if len(flat_wins):
    display(flat_wins[["id", "question", "ground_truth", "graph_rag_answer"]])

## Streamlit Demo UI

```bash
streamlit run streamlit_app.py
```

Tabs: **Query** (so sánh Flat vs Graph) | **Graph** (visualize + BFS) | **Evaluation** (20 câu benchmark) | **Corpus**

## (Tùy chọn) Neo4j & NodeRAG

- **Neo4j**: `python main.py --neo4j` (cần Neo4j Desktop + password trong `.env`)
- **NodeRAG**: `python noderag_setup.py` rồi `pip install NodeRAG && python -m NodeRAG.build -f noderag_project`